<a href="https://colab.research.google.com/github/Vaib-raksh/Intern-ML/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vaib-raksh/Intern-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

#Unit of Analysis

One row represents the SEO performance of a single webpage for a specific reporting period. Each row contains metrics such as CTR, impressions, search position, and other SEO-related information for that webpage.

### Time Window

For this assignment, I will use a mid-panel month (March 2026) for analysis and feature engineering. I am not using the final month (June 2026) because it should remain as unseen data for honest evaluation and to avoid using future information while developing the analysis.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

#Features
- position_tier
- impressions_90d
- search_volume
- content_type

These features are available before making a content optimization decision and may help estimate CTR.

##Label
- ctr

CTR is the target value that I want to predict.

### Context
- month
- date (or reporting period)
- webpage identifier (if available)

These fields provide context about when and which webpage the data represents but are not the prediction target.

### Excluded
- Final month (June 2026)

I excluded the final month because it should remain as unseen data for honest evaluation. Any field that directly reveals the target would also be excluded to avoid data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [3]:
!pip install -q duckdb datasets huggingface_hub   #connecting the dataset- install the required libraries

In [4]:
# Read the HF_TOKEN from colab
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("Token loaded successfully!" if HF_TOKEN else "Token not found")

Token loaded successfully!


In [5]:
#connect DuckDB to Hugging face
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
);
""")

print("Connected successfully!")

Connected successfully!


In [6]:
DATASET = "hf://datasets/FlyRank/internship-warehouse" # the dataset path

In [7]:
# Checking for tables and columns present in the dataset
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{DATASET}/fact_content_daily_performance/**/*.parquet'
)
LIMIT 5
""")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [8]:
# Test the connection
con.sql(f"""
SELECT COUNT(*)
FROM read_parquet(
    '{DATASET}/fact_content_daily_performance/**/*.parquet'
)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [9]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks
FROM read_parquet(
    '{DATASET}/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 10
""")

┌─────────────┬─────────────────────────┬──────────────────────────┬─────────────────┬────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ gsc_impressions │ gsc_clicks │
│    date     │         varchar         │         varchar          │      int64      │   int64    │
├─────────────┼─────────────────────────┼──────────────────────────┼─────────────────┼────────────┤
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │              20 │          0 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_05597932fe4da067 │               1 │          0 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_7a105f548d9c6916 │             125 │          1 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_905aa32a0230694e │               7 │          0 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_a3ea9792f793ec72 │              11 │          0 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_36c36abc7650d7af │             239 │          1 │


In [10]:
con.sql(f"""
SELECT COUNT(*)

FROM read_parquet(
'{DATASET}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      9841378 │
└──────────────┘

In [11]:
# Query- 1 Analysis(Grain)
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks
FROM read_parquet(
    '{DATASET}/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 10
""")

┌─────────────┬─────────────────────────┬──────────────────────────┬─────────────────┬────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ gsc_impressions │ gsc_clicks │
│    date     │         varchar         │         varchar          │      int64      │   int64    │
├─────────────┼─────────────────────────┼──────────────────────────┼─────────────────┼────────────┤
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │              20 │          0 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_05597932fe4da067 │               1 │          0 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_7a105f548d9c6916 │             125 │          1 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_905aa32a0230694e │               7 │          0 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_a3ea9792f793ec72 │              11 │          0 │
│ 2026-03-01  │ client_73cda7b4e4f265ea │ content_36c36abc7650d7af │             239 │          1 │


#Verified: Query 1
By inspecting sample records, one row represents the daily performance of a single content item for one client on one reporting date. Each row contains the SEO metrics (such as impressions and clicks) for that content on that day.

In [13]:
#Query 2 Analysis (COUNTS)
con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet(
    '{DATASET}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│    9841378 │
└────────────┘

#Verified: Query-2
The March 2026 partition contains 9,841,378 rows, providing a large dataset for analyzing relationships between search performance metrics and CTR.

In [14]:
#Query 3 Analysis (MISSING VALUES)
con.sql(f"""
SELECT COUNT(*) AS usable_rows
FROM read_parquet(
    '{DATASET}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┐
│ usable_rows │
│    int64    │
├─────────────┤
│     3611061 │
└─────────────┘

#Verified: Query 3
After filtering with gsc_data_available IS TRUE, 3,611,061 rows remain. These are the rows used for analyses that require Google Search Console metrics.

In [15]:
# Query 4 (WINDOW)
con.sql(f"""
SELECT
    MIN(report_date) AS first_day,
    MAX(report_date) AS last_day
FROM read_parquet(
    '{DATASET}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┐
│ first_day  │  last_day  │
│    date    │    date    │
├────────────┼────────────┤
│ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┘

#Verified: Query 4
The selected partition covers the complete March 2026 time window, from 2026-03-01 to 2026-03-31.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limits- ANSWER

This dataset has some limitations. Different clients have different amounts of historical data, so long-term comparisons may not be equally reliable for every client. Some rows may not contain complete Google Search Console data, which reduces the number of usable observations for analysis. This analysis can only identify observed relationships between position and CTR. It cannot prove that moving a page to a higher position will always cause its CTR to increase, because many other factors can influence user behavior and search performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.